# Querying datasets using Google Earth Engine API
This file provides functions to query various spatial data sets using the Google Earth Engine API. Also included is the code to complete the authentication and initialization, example usages of the function, brief explanations of the function, and descriptions of the datasets for which the function may be used.

First, the API must be authenticated and initialized. To do so, the code below must be run. This will open a webpage where a token can be generated. The resulting verification code should be pasted in the notebook cell output. Before running the code, ensure the project specified in the code below matches the Google Clod project to be used. 

In [ ]:
# Authenticate and initialize Google Earth Engine in a Jupyter Notebook
import ee
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='my-project-id') # Replace 'my-project-id' with your actual Google Cloud project ID

In [2]:
# Import the necessary libraries
import geemap.core as geemap
import pandas as pd

## General, generic function to query datasets
Below is a function to collect data from the Google Earth Engine API from a select number of datasets, for a point and year to be specified by the user. Possible datasets are outlined and described below. 

The function takes as input the latitude and longitude coordinates of the target point, the target year, the dataset ID and variable for which to extract data. There are also parameters to specify a buffer distance (in meters) of the radius around the given coordinates to search, and a spatial aggregation method (possible input: mean/max/min/median/std) that specifies how data in this region is to be aggregated. Users can also specify a time window around the given year (if 0, only the target year is considered), and a temporal aggregation method (possible input: closest/mean/max/min/std/sum). If 'closest' is specified as the temporal aggregation method, the closest single image to the date given (within the time window) will be selected from which to retrieve data. In these cases, a boolean parameter 'closest_precedent' can be used to specify whether to query the closest image in the years preceding the given year ('True') or the absolute closest image ('False'). If 'mean', 'max', 'min', or 'std' are specified as the temporal aggregation method, all images in the time window specified will be used to find the corresponding aggregated value.

The function returns the value found, the date corresponding to this value, and the difference between the result year and the target year. If no image is found within the time window, 'None' will be returned; in this case, consider widening the time window.

Both Image Collection datasets and Static Image datasets can be queried using this generic function; the generic function first identifies the type of asset, then calls one of the two 'helper' functions, according to the type.

In [107]:
def query_static_image(
    lat,
    lon,
    dataset_id,
    variable_names,
    site_code=None,
    buffer_meters=0,
    spatial_agg='mean',
    year=None  
):
    
    """
    Takes as input a latitude, longitude, dataset ID, variable names, site code, buffer in meters, spatial aggregation method (mean/max/min/median/std), and year to be queried.
    Queries a static GEE Image at the point and year given (or around that point, depending on the given buffer in meters and aggregation method).
    Returns a list with the value for the given variables in the given dataset, the date, and the year difference.
    """

    # Progress print statement
    print(f"Querying static image dataset '{dataset_id}' at ({lat}, {lon}), year {year} with variables {variable_names}...")

    # Identify the point to be queried (with buffer if specified)
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(buffer_meters) if buffer_meters > 0 else point

    # Prepare the result dictionary
    result = {}

    # Map spatial aggregation methods to Earth Engine reducers
    agg_map = {
        'mean': ee.Reducer.mean(),
        'max': ee.Reducer.max(),
        'min': ee.Reducer.min(),
        'median': ee.Reducer.median(),
        'std': ee.Reducer.stdDev(),
        'sum': ee.Reducer.sum()
    }

    # Use the specified spatial aggregation method or default to mean (if not found in agg_map)
    reducer = agg_map.get(spatial_agg, ee.Reducer.mean())

    # Load the static image dataset and select the specified variable names
    image = ee.Image(dataset_id)

    # Get the band names from the image
    band_names = image.bandNames().getInfo()

    # Check if the dataset is Hansen Global Forest Change
    is_hansen = dataset_id == "UMD/hansen/global_forest_change_2024_v1_12"

    # Initialize variables for year difference and date to use
    year_diff = None
    date_to_use = "unknown"

    # Handle datasets with timesteps for band names
    if len(band_names) >= 1 and all(b.startswith('b') for b in band_names) and year is not None:
        start_year = 1990
        end_year = start_year + len(band_names) - 1
        closest_year = max(min(year, end_year), start_year)
        band_index = closest_year - start_year + 1
        band_name = f"b{band_index}"

        variable_names = [band_name]
        image = image.select(variable_names)
        year_diff = closest_year - year
        date_to_use = str(closest_year)

    # Handle Hansen Global Forest Change dataset
    elif is_hansen and 'lossyear' in band_names and year is not None:

        # Check if the year is within the valid range for Hansen dataset
        if year < 2001 or year > 2024:
            raise ValueError(f"'lossyear' only valid from 2001–2024. Got: {year}")

        # Add binary mask for this year's forest loss
        year_offset = ee.Number(year).subtract(2000)
        loss_mask = image.select('lossyear').eq(year_offset).rename('loss_in_year')
        image = image.addBands(loss_mask)

        variable_names_with_loss = list(variable_names) 
        if 'loss_in_year' not in variable_names_with_loss:
            variable_names_with_loss.append('loss_in_year')

        image = image.select(variable_names_with_loss).unmask(0)
        date_to_use = str(year)
        year_diff = 0

    # Handle default case
    else:
        image = image.select(variable_names)

        # For default case: if date metadata is available, extract the start date and calculate year difference
        try:
            date_range = ee.List(image.get('date_range'))
            start_ms = ee.Number(date_range.get(0))
            start_date_obj = ee.Date(start_ms)
            date_to_use = str(start_date_obj.get('year').getInfo())
            year_diff = int(date_to_use) - year if year is not None else None

        # For default case: if start date is not found is not available, default to "unknown" 
        except Exception:
            date_to_use = "unknown"

    # Get the image details for the specified point or region with buffer, and variables
    stats = image.reduceRegion(
        reducer=reducer,
        geometry=region,
        scale=30,
        maxPixels=1e9
    ).getInfo()

    # Fill the result dictionary with variable values, dates, and year difference
    for var in variable_names:

        key_base = f"{var}_buffer{buffer_meters}_spatial{spatial_agg}"

        result[f"{key_base}_value"] = stats.get(var, None)
        result[f"{key_base}_date"] = date_to_use

    result['year_diff'] = year_diff

    # Return the result dictionary
    return result

# Function to query Image Collection datasets
def query_image_collection(
    lat,
    lon,
    year,
    dataset_id,
    variable_names,
    site_code=None,
    country_code=None,
    buffer_meters=0,
    spatial_agg='mean',
    temporal_agg='mean',
    time_window_years=1,
    closest_precedent=True
):
    
    """
    Takes as input a latitude, longitude, the year to be queried, dataset ID, variable names, site code, buffer in meters, spatial aggregation method (mean/max/min/median/std), 
    temporal aggregation method (closest/mean/min/max/std), time window in years, the maximum year difference, and a closest precedent flag.
    Queries an Image Collection at the point and year given (or around that point/time, depending on the given buffer in meters, time window, and aggregation methods).
    Returns a list with the value for the given variables in the given dataset, the date, and the year difference.
    """

    # Progress print statement
    print(f"Querying image collection '{dataset_id}' at ({lat}, {lon}), year {year} with variables {variable_names}...")

    # Import necessary libraries
    import numpy as np

    # Identify the point to be queried (with buffer if specified)
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(buffer_meters) if buffer_meters > 0 else point

    # Initialize the result dictionary
    result = {}

    # Ensure the dataset is an ImageCollection with metadata for the start time
    def ensure_time_start(col):
        def add_time(img):
            time_start = img.get('system:time_start')
            system_index = ee.String(img.get('system:index'))
            parts = system_index.split('_')
            date_str = ee.Algorithms.If(
                parts.length().gt(1),
                parts.get(1),
                system_index  
            )
            return ee.Algorithms.If(
                time_start,
                img,
                ee.Image(img).set(
                    'system:time_start',
                    ee.Date(ee.String(date_str)).millis()
                )
            )
        return ee.ImageCollection(col.map(lambda i: ee.Image(add_time(i))))
    
    # Filter WorldPop by point location first, or get ImageCollection for all other datasets
    if dataset_id.startswith("WorldPop/GP/100m/pop") and country_code:
        # Take all images for country code
        image_ids = [
            f"{dataset_id}/{country_code.upper()}_{y}"
            for y in range(2000, 2021)
        ]
        images = [ee.Image(id) for id in image_ids]
        collection = ee.ImageCollection(images)
    else:
        collection = ensure_time_start(ee.ImageCollection(dataset_id))

    # Filter the collection by date (with time window if specified)
    start_date = ee.Date.fromYMD(year - time_window_years, 1, 1)
    end_date = ee.Date.fromYMD(year + time_window_years, 12, 31)
    filtered_col = collection.filterDate(start_date, end_date)

    # Find number of images in the filtered collection
    n_imgs_filtered = filtered_col.size().getInfo()
    n_imgs_total = collection.size().getInfo()

    # Handle cases with no images in the filtered collection
    # If there is only one, take that image
    if n_imgs_filtered == 0 and n_imgs_total == 1:
        single_img = ee.Image(collection.first())

        # Try to find the date from metadata
        try:
            date_range = ee.List(single_img.get('date_range'))
            start_ms = ee.Number(date_range.get(0))
            start_date_obj = ee.Date(start_ms)
            img_year = start_date_obj.get('year').getInfo()
            date_str = start_date_obj.format('YYYY-MM-dd').getInfo()
            year_diff = img_year - year

        # If metadata is not available, fallback to finding the year elsewhere
        except Exception:
            system_index = single_img.get('system:index').getInfo()
            import re
            match = re.search(r'\b(19|20)\d{2}\b', system_index)
            if match:
                img_year = int(match.group(0))
                date_str = f"{img_year}-01-01"
                year_diff = img_year - year
            else:
                date_str = "unknown"
                year_diff = None
                img_year = None

        # Use the specified spatial aggregation method or default to mean (if inputted value is not found)
        reducer = {
            'mean': ee.Reducer.mean(),
            'max': ee.Reducer.max(),
            'min': ee.Reducer.min(),
            'median': ee.Reducer.median(),
            'std': ee.Reducer.stdDev(),
            'sum': ee.Reducer.sum()
        }.get(spatial_agg, ee.Reducer.mean())

        # Get the image details for the specified point or region with buffer, and variables
        stats = single_img.select(variable_names).reduceRegion(
            reducer=reducer,
            geometry=region,
            scale=30,
            maxPixels=1e9
        ).getInfo()

        # Fill the result dictionary with variable values, dates, and year difference
        for v in variable_names:

            key_base = f"{v}_buffer{buffer_meters}_years{time_window_years}_spatial{spatial_agg}_temporal{temporal_agg}"

            result[f"{key_base}_value"] = stats.get(v, None)
            result[f"{key_base}_date"] = date_str

        result["year_diff"] = year_diff

        # Return the result dictionary
        return result

    # If there are no images in the filtered collection, return None for all variables
    if n_imgs_filtered == 0:
        for v in variable_names:

            key_base = f"{v}_buffer{buffer_meters}_years{time_window_years}_spatial{spatial_agg}_temporal{temporal_agg}"

            result[f"{key_base}_value"] = None
            result[f"{key_base}_date"] = None
        result["year_diff"] = None
        return result

    # If there are several images, convert the filtered collection to a list
    imgs_list = filtered_col.toList(n_imgs_filtered)

    # If temporal aggregation is 'closest', find the closest image (single date) to the target year
    if temporal_agg == 'closest':

        # Initialize variables to track the closest image
        closest_img = None
        closest_diff = None
        closest_date = None
        closest_year = None

        # Iterate through the images to find the closest one
        for i in range(n_imgs_filtered):
            img = ee.Image(imgs_list.get(i))
            date = ee.Date(img.get('system:time_start'))
            img_year = date.get('year').getInfo()
            diff = img_year - year

            # If searching for subequent images, skip images that are before the target year
            if closest_precedent and diff > 0:
                continue

            # Calculate the absolute difference
            abs_diff = abs(diff)

            # If this is the first image or closer than the previous closest, update
            if closest_img is None or abs_diff < closest_diff:
                closest_img = img
                closest_diff = abs_diff
                closest_date = date
                closest_year = img_year

        # If no image is found within the time window, return None for all variables
        if closest_diff is None:
            for v in variable_names:

                key_base = f"{v}_buffer{buffer_meters}_years{time_window_years}_spatial{spatial_agg}_temporal{temporal_agg}"

                result[f"{key_base}_value"] = None
                result[f"{key_base}_date"] = None
            result["year_diff"] = None
            return result

        # Use the specified spatial aggregation method or default to mean (if inputted value is not found)
        reducer = {
            'mean': ee.Reducer.mean(),
            'max': ee.Reducer.max(),
            'min': ee.Reducer.min(),
            'median': ee.Reducer.median(),
            'std': ee.Reducer.stdDev(),
            'sum': ee.Reducer.sum()
        }.get(spatial_agg, ee.Reducer.mean())

        # Get the image details (for given variables) for the closest image at the point or region with buffer
        stats = closest_img.select(variable_names).reduceRegion(
            reducer=reducer,
            geometry=region,
            scale=30,
            maxPixels=1e9
        ).getInfo()

        # Fill the result dictionary with variable values, date, and year difference
        date_str = closest_date.format('YYYY-MM-dd').getInfo()

        for v in variable_names:

            key_base = f"{v}_buffer{buffer_meters}_years{time_window_years}_spatial{spatial_agg}_temporal{temporal_agg}"

            result[f"{key_base}_value"] = stats.get(v, None)
            result[f"{key_base}_date"] = date_str
        result["year_diff"] = closest_year - year

        # Return the result dictionary
        return result

    # If not looking for closest image, but aggregating over year range (mean, min, max, std), initalize lists to store values and years
    values_by_image = []
    image_years = []

    # Iterate through the images in the filtered collection
    for i in range(n_imgs_filtered):
        img = ee.Image(imgs_list.get(i))
        date = ee.Date(img.get('system:time_start'))
        img_year = date.get('year').getInfo()
        image_years.append(img_year)

        # Get the image details (for given variables) for the images at the point or region with buffer
        stats = img.select(variable_names).reduceRegion(
            reducer={
                'mean': ee.Reducer.mean(),
                'max': ee.Reducer.max(),
                'min': ee.Reducer.min(),
                'median': ee.Reducer.median(),
                'std': ee.Reducer.stdDev(),
                'sum': ee.Reducer.sum()
            }.get(spatial_agg, ee.Reducer.mean()),
            geometry=region,
            scale=30,
            maxPixels=1e9
        ).getInfo()
        values_by_image.append(stats)

    # Find the actual year based on the median of the image years
    actual_year = int(np.median(image_years)) if image_years else None

    # Iterate thorough the variable names and aggregate values across images according to the specified temporal aggregation method (if not 'closest')
    for v in variable_names:

        key_base = f"{v}_buffer{buffer_meters}_years{time_window_years}_spatial{spatial_agg}_temporal{temporal_agg}"

        vals = [d[v] for d in values_by_image if d.get(v) is not None]
        if not vals:
            result[f"{key_base}_value"] = None
            result[f"{key_base}_date"] = None
        else:
            arr = np.array(vals, dtype=float)
            if temporal_agg == 'mean':
                agg_val = np.mean(arr)
            elif temporal_agg == 'min':
                agg_val = np.min(arr)
            elif temporal_agg == 'max':
                agg_val = np.max(arr)
            elif temporal_agg == 'std':
                agg_val = np.std(arr)
            else:
                agg_val = np.mean(arr)
            result[f"{key_base}_value"] = float(agg_val)
            result[f"{key_base}_date"] = str(actual_year)

    # Fill the year difference in the result
    result['year_diff'] = actual_year - year if actual_year is not None else None

    # Return the result dictionary
    return result

# Function to query datasets generically, using the appropriate function based on dataset type
def query_dataset_generic(
    lat,
    lon,
    year,
    dataset_id,
    variable_names,
    site_code=None,
    country_code=None,
    buffer_meters=0,
    time_window_years=1,
    closest_precedent=True,
    spatial_agg='mean',
    temporal_agg='mean' 
):
    """
    Takes as input a latitude, longitude, year to be queried, dataset ID, variable names, site code, buffer in meters, time window in years, a closest precedent flag,
    a spatial aggregation method (mean/max/min/median/std), and a temporal aggregation method (closest/mean/min/max/std).
    Identifies asset type and query accordingly either a static Image or ImageCollection dataset by calling on one of two functions.
    """

    # Identify the asset type of the dataset
    asset_info = ee.data.getInfo(dataset_id)
    asset_type = asset_info.get('type', '')

    # If asset type is an Image Collection, query it as an Image Collection
    if asset_type == 'IMAGE_COLLECTION' or asset_type == 'FOLDER':
        return query_image_collection(
            lat, lon, year, dataset_id, variable_names,
            site_code=site_code,
            country_code=country_code,
            buffer_meters=buffer_meters,
            time_window_years=time_window_years,
            closest_precedent=closest_precedent,
            spatial_agg=spatial_agg,
            temporal_agg=temporal_agg
        )
    
    # If asset type is an Image, query it as a Static Image
    elif asset_type == 'IMAGE':
        return query_static_image(
            lat, lon, dataset_id, variable_names,
            site_code=site_code,
            buffer_meters=buffer_meters,
            spatial_agg=spatial_agg,
            year=year
        )
    
    # Raise an error if the dataset is neither an Image nor an Image Collection
    else:
        raise ValueError(f"Dataset {dataset_id} is not an Image or Image Collection.")

## Datasets
* Global Human Modification:
    * Dataset: CSP gHM: Global Human Modification (dataset ID: 'CSP/HM/GlobalHumanModification')
    * Variables: Global Human Modification representing the degree to which terrestrial lands have been modified by humans on a scale of 0.0 to 1.0 (variable name: 'gHM')
    * Coverage: 1 year (2016)
    * Dataset Provider: Conservation Science Partners
    * Citations: 
        * Kennedy, C.M., J.R. Oakleaf, D.M. Theobald, S. Baruch-Murdo, and J. Kiesecker. 2019. Managing the middle: A shift in conservation priorities based on the global human modification gradient. Global Change Biology 00:1-16. doi:10.1111/gcb.14549
* Human Impact Index 
    * Datasets: 
        * Human Impact Index (dataset ID: 'projects/HII/v1/hii')
        * Infrastructure (dataset ID: 'projects/HII/v1/driver/infrastructure')
        * Land Use (dataset ID: 'projects/HII/v1/driver/land_use')
        * Population Density (dataset ID: 'projects/HII/v1/driver/population_density')
        * Power (dataset ID: 'projects/HII/v1/driver/power')
        * Railways (dataset ID: 'projects/HII/v1/driver/railways')
        * Roads (dataset ID: 'projects/HII/v1/driver/roads')
        * Water (dataset ID: 'projects/HII/v1/driver/water')
    * Variables: 
        * For Human Impact Index: Human Impact Index (variable name: 'hii')
        * For Infrastructure: Infrastructure (variable name: 'hii_infrastucture_driver')
        * For Land Use: Land Use (variable name: 'hii_landuse_driver')
        * For Population Density: Population Density (variable name: 'hii_popdens_driver')
        * For Power: Power (variable name: 'hii_power_driver')
        * For Railways: Railways (variable name: 'hii_railway_driver')
        * For Roads: Roads (variable name: 'hii_road_driver')
        * For Water: Water (variable name: 'hii_water_driver')
    * Coverage: 19 years (2021-2020)
    * Dataset Provider: Wildlife Conservation Society
* Global Human Settlement Layer
    * Dataset: GHSL: Global settlement characteristics (10 m) 2018 (P2023A) (dataset ID: 'JRC/GHSL/P2023A/GHS_BUILT_C')
    * Variables: Settlement characteristics (variable name: 'built_characteristics') with ordinal-ish values!
    * Coverage: 1 year (2018)
    * Dataset: GHSL: Global built-up surface 1975-2030 (dataset ID: 'JRC/GHSL/P2023A/GHS_BUILT_S')
    * Variables: built-up surfaces and non-residential built-up surfaces in square metres per 100 m grid cell (variable names: 'built_surface', 'built_surface_nres')
    * Coverage: 5-year intervals between 1975 and 2030
    * Dataset Provider: EC JRC
    * Citations: 
        * Pesaresi, Martino; Politis, Panagiotis (2023): GHS-BUILT-C R2023A - GHS Settlement Characteristics, derived from Sentinel2 composite (2018) and other GHS R2023A data. European Commission, Joint Research Centre (JRC) PID: http://data.europa.eu/89h/3c60ddf6-0586-4190-854b-f6aa0edc2a30 doi:10.2905/3c60ddf6-0586-4190-854b-f6aa0edc2a30
        * Pesaresi, Martino, Marcello Schiavina, Panagiotis Politis, Sergio Freire, Katarzyna Krasnodebska, Johannes H. Uhl, Alessandra Carioli, et al. (2024). Advances on the Global Human Settlement Layer by Joint Assessment of Earth Observation and Population Survey Data. International Journal of Digital Earth 17(1). doi:10.1080/17538947.2024.2390454.
* WorldPop Global Population Data
    * Dataset: WorldPop Global Project Population Data: Estimated Residential Population per 100x100m Grid Square (dataset ID: 'WorldPop/GP/100m/pop')
    * Variables: Estimated number of people residing in each grid cell (variable name: 'population')
    * Coverage: 21 years (2000-2021)
    * Dataset Provider: WorldPop (www.worldpop.org)
    * Citations: 
        * Americas population data: Alessandro Sorichetta, Graeme M. Hornby, Forrest R. Stevens, Andrea E. Gaughan, Catherine Linard, Andrew J. Tatem, 2015, High-resolution gridded population datasets for Latin America and the Caribbean in 2010, 2015, and 2020, Scientific Data, doi:10.1038/sdata.2015.45
        * Africa population count data: Linard, C., Gilbert, M., Snow, R.W., Noor, A.M. and Tatem, A.J., 2012, Population distribution, settlement patterns and accessibility across Africa in 2010, PLoS ONE, 7(2): e31743.
        * Asia population count data: Gaughan AE, Stevens FR, Linard C, Jia P and Tatem AJ, 2013, High resolution population distribution maps for Southeast Asia in 2010 and 2015, PLoS ONE, 8(2): e55882.
* GPW Population Density
    * Dataset: GPWv411: Population Density (Gridded Population of the World Version 4.11) (dataset ID: 'CIESIN/GPWv411/GPW_Population_Density')
    * Variables: The estimated number of persons per square kilometer (variable name: 'population_density')
    * Coverage: 20 years (2000-2020) 
    * Dataset Provider: NASA SEDAC at the Center for International Earth Science Information Network
    * Citations:
        * Center for International Earth Science Information Network - CIESIN - Columbia University. 2018. Gridded Population of the World, Version 4 (GPWv4): Population Density, Revision 11. Palisades, NY: NASA Socioeconomic Data and Applications Center (SEDAC). https://doi.org/10.7927/H49C6VHW. Accessed 26 June 2025.
* Global Travel Time
    * Dataset: Accessibility to Cities (dataset ID: 'Oxford/MAP/accessibility_to_cities_2015_v1_0')
    * Variables: Travel time to the nearest densely-populated area (variable name: 'accessibility')
    * Coverage: 1 year (2015)
    * Dataset Provider: Malaria Atlas Project
    * Citations:
        * D.J. Weiss, A. Nelson, H.S. Gibson, W. Temperley, S. Peedell, A. Lieber, M. Hancher, E. Poyart, S. Belchior, N. Fullman, B. Mappin, U. Dalrymple, J. Rozier, T.C.D. Lucas, R.E. Howes, L.S. Tusting, S.Y. Kang, E. Cameron, D. Bisanzio, K.E. Battle, S. Bhatt, and P.W. Gething. A global map of travel time to cities to assess inequalities in accessibility in 2015. Nature (2018). doi:10.1038/nature25181
* Rural Access Index
    * Note: this is focused on *rural* access, so doesn't include some spaces presumably coded as *urban*. Inaccessibility mask also treats many cells as NA, which can impact some summary stats (e.g., near rivers or near urban spaces).
    * Dataset: 
        * Rural Access Index (RAI) - RAI Multiplier (dataset ID: 'projects/sat-io/open-datasets/RAI/raimultiplier')
        * Rural Access Index (RAI) - Rural Pop (dataset ID:'projects/sat-io/open-datasets/RAI/ruralpop')
        * Rural Access Index (RAI) - Rural Pop Access (dataset ID: 'projects/sat-io/open-datasets/RAI/ruralpopaccess')
    * Variables:
        * For RAI Multiplier: Inaccessibility Index (variable name: 'b1')
        * For Rural Pop: Rural Population (variable name: 'population')
        * For Rural Pop Access: Rural Population with access (variable name: 'population')
    * Coverage: 1 year (updated in 2024)
    * Dataset Provider: United Nations Sustainable Development Solutions Network
    * Citations:
        * Iablonovski G, Drumm E, Fuller G and Lafortune G (2024) A global implementation of the rural access index. Front. Remote Sens. 5:1375476. doi: 10.3389/frsen.2024.1375476
* Gross Domestic Product and Human Development Index
    * Dataset: 
        * GDP per capita (PPP) (dataset ID: 'projects/earthengine-legacy/assets/projects/sat-io/open-datasets/GRIDDED_HDI_GDP/GDP_per_capita_PPP_1990_2015_v2')
        * HDI (dataset ID: 'projects/sat-io/open-datasets/GRIDDED_HDI_GDP/HDI_1990_2015_v2')
    * Variables:
        * For GDP pe capita (PPP): Gridded GDP per capita, derived from a combination of sub-national and national datasets - each band is a timestep, representing a year between 1990 and 2015 (variable names: 'b1' to 'b26')
        * For HDI: Gridded HDI, derived from a combination of sub-national and national datasets - each band is a timestep, representing a year between 1990 and 2015 (variable names: 'b1' to 'b26')
    * Coverage: 25 years (1990-2015)
    * Dataset Provider: Kummu, Matti; Taka, Maija; Guillaume, Joseph H. A.
    * Citations:
        * Kummu, M., Taka, M. & Guillaume, J. Gridded global datasets for Gross Domestic Product and Human Development Index over 1990–2015. Sci Data 5, 180004 (2018). https://doi.org/10.1038/sdata.2018.4
* TerraClimate climate and water balance databases
    * Dataset: TerraClimate: Monthly Climate and Climatic Water Balance for Global Terrestrial Surfaces, University of Idaho (dataset ID: 'IDAHO_EPSCOR/TERRACLIMATE')
    * Variables: 
        * Actual evapotranspiration in millimeters, as measured using a one-dimensional soil water balance model (variable name: 'aet')
        * Climate water deficit in millimeters, as measured using a one-dimensional soil water balance model (variable name: 'def')
        * Palmer Drought Severity Index (variable name: 'pdsi')
        * Reference evapotranspiration in millimeters (variable name: 'pet')
        * Precipitation accumulation in millimeters (variable name: 'pr')
        * Runoff, as measured using a one-dimensional soil water balance model in millimeters (variable name: 'ro')
        * Soil moisture in millimeters, as measuring using a one-dimensional soil water balance model (variable name: 'soil')
        * Downward surface shortwave radiation in W/m^2 (variable name: 'srad')
        * Snow water equivalent in millimeters, as measured using a one-dimensional soil water balance model (variable name: 'swe')
        * Minimum temperature in °C (variable name: 'tmmn')
        * Maximum temperature in °C (variable name: 'tmmx')
        * Vapor pressure in kPa (variable name: 'vap')
        * Vapor pressure deficit in kPa (variable name: 'vpd')
        * Wind-speed at 10m in m/s (variable name: 'vs')
    * Coverage: 66 years (1958-2024)
    * Dataset Provider: University of California Merced
    * Citations:
        * Abatzoglou, J.T., S.Z. Dobrowski, S.A. Parks, K.C. Hegewisch, 2018, Terraclimate, a high-resolution global dataset of monthly climate and climatic water balance from 1958-2015, Scientific Data 5:170191, doi:10.1038/sdata.2017.191
* GRACE Drought Severity Index
    * Dataset: GRACE Monthly Mass Grids Version 04 - Global Mascon (CRI Filtered) (dataset ID: 'NASA/GRACE/MASS_GRIDS_V04/MASCON_CRI')
    * Variables: 
        * Equivalent liquid water thickness in centimeters (variable name: 'lwe_thickness') 
        * 1-sigma uncertainty for each 3-degree mascon estimate (regarded as conservative) (variable name: 'uncertainty')
        * Note: need to derive from this the actual GRACE Drought Severity Index! 
    * Coverage: 22 years (2002-2024)
    * Dataset Provider: NASA Jet Propulsion Laboratory
    * Citations:
        * D. N. Wiese, D.-N. Yuan, C. Boening, F. W. Landerer, M. M. Watkins. 2023. JPL GRACE and GRACE-FO Mascon Ocean, Ice, and Hydrology Equivalent Water Height CRI Filtered RL06.3Mv04. Ver. RL06.3Mv04. PO.DAAC, CA, USA. Dataset accessed [2025-06-26] at https://doi.org/10.5067/TEMSC-3JC634.
        * Watkins, M. M., D. N. Wiese, D.-N. Yuan, C. Boening, and F. W. Landerer (2015), Improved methods for observing Earth's time variable mass, mass distribution with GRACE using spherical cap mascons, J. Geophys. Res Solid Earth, 120, doi:10.1002/2014JB011547.
        * Wiese, D. N., F. W. Landerer, and M. M. Watkins (2016), Quantifying and reducing leakage errors in the JPL RL05M GRACE mascon solution, Water Resour. Res., 52, 7490-7502, doi:10.1002/2016WR019344.
* SPEI Global Drought Monitor data
    * Dataset: SPEIbase: Standardised Precipitation-Evapotranspiration Index database, Version 2.10 (dataset ID: 'CSIC/SPEI/2_10')
    * Variables:
        * Standardized Precipitation-Evapotranspiration Index (SPEI) where precipitation and evapotranspiration data was accumulated over the previous month (variable name: 'SPEI_01_month')
        * Standardized Precipitation-Evapotranspiration Index (SPEI) where precipitation and evapotranspiration data was accumulated over the previous 2 months (variable name: 'SPEI_02_month')
        * ...
        * Standardized Precipitation-Evapotranspiration Index (SPEI) where precipitation and evapotranspiration data was accumulated over the previous 48 months (variable name: 'SPEI_48_month')
    * Coverage: 122 years (1901-2023)
    * Dataset Provider: Spanish National Research Council (CSIC)
    * Citations: 
        * Reig-Gracia, Fergus; Latorre Garcés, Borja; 2023; SPEIbase v.2.9 [Dataset]; DIGITAL.CSIC; Version 2.9. doi:10.20350/digitalCSIC/15470
        * Vicente-Serrano S.M., Beguería S., López-Moreno J.I. (2010): A Multi-scalar drought index sensitive to global warming: The Standardized Precipitation Evapotranspiration Index - SPEI. Journal of Climate 23(7), 1696-1718. doi:10.1175/2009JCLI2909.1
* Global Forest Change
    * Dataset: Hansen Global Forest Change v1.12 (2000-2024) (dataset ID: 'UMD/hansen/global_forest_change_2024_v1_12')
    * Variables:
        * Tree canopy cover for year 2000 in %, defined as canopy closure for all vegetation taller than 5m in height (variable name: 'treecover2000')
        * Binary indicator for forest loss during the study period (0: Not loss; 1: Loss), defined as a stand-replacement disturbance (a change from a forest to non-forest state) (variable name: 'loss')
        * Binary indicator of forest gain during the period 2000-2012 (0: No gain; 1: Gain), defined as the inverse of loss (a non-forest to forest change entirely within the study period) (variable name: 'gain')
        * Year of gross forest cover loss event, encoded as an integer (0 for no loss, or a value in the range 1-24 representing loss detected primarily in the year 2001-2024, respectively) (variable name: 'lossyear')
        * Landsat Red cloud-free image composite (corresponding to Landsat 5/7 band 3, 4, 5, and 7 and Landsat 8/9 band 4, 5, 6, and 7). Reference multispectral imagery from the first available year, typically 2000 (variable name: 'first_b30')
        * Landsat NIR cloud-free image composite (corresponding to Landsat 5/7 band 4 and Landsat 8/9 band 5). Reference multispectral imagery from the first available year, typically 2000 (variable name: 'first_b40')
        * Landsat SWIR1 cloud-free image composite (corresponding to Landsat 5/7 band 5 and Landsat 8/9 band 6). Reference multispectral imagery from the first available year, typically 2000 (variable name: 'first_b50')
        * Landsat SWIR2 cloud-free image composite (corresponding to Landsat 5/7 band 7 and Landsat 8/9 band 7). Reference multispectral imagery from the first available year, typically 2000 (variable name: 'first_b70')
        * Landsat Red cloud-free image composite (corresponding to Landsat 5/7 band 3 and Landsat 8/9 band 4). Reference multispectral imagery from the last available year, typically the last year of the study period (variable name: 'last_b30')
        * Landsat NIR cloud-free image composite (corresponding to Landsat 5/7 band 4 and Landsat 8/9 band 5). Reference multispectral imagery from the last available year, typically the last year of the study period (variable name: 'last_b40')
        * Landsat SWIR1 cloud-free image composite (corresponding to Landsat 5/7 band 5 and Landsat 8/9 band 6). Reference multispectral imagery from the last available year, typically the last year of the study period (variable name: 'last_b50')
        * Landsat SWIR2 cloud-free image composite (corresponding to Landsat 5/7 band 7 and Landsat 8/9 band 7). Reference multispectral imagery from the last available year, typically the last year of the study period (variable name: 'last_b70')
        * Three values representing areas of no data (0), mapped land surface (1), and permanent water bodies (2) (variable name: 'datamask')
    * Coverage: 24 years (2000-2024)
    * Dataset Provider: Hansen/UMD/Google/USGS/NASA
    * Citations: 
        * Hansen, M. C., P. V. Potapov, R. Moore, M. Hancher, S. A. Turubanova, A. Tyukavina, D. Thau, S. V. Stehman, S. J. Goetz, T. R. Loveland, A. Kommareddy, A. Egorov, L. Chini, C. O. Justice, and J. R. G. Townshend. 2013. "High-Resolution Global Maps of 21st-Century Forest Cover Change." Science 342 (15 November): 850-53. 10.1126/science.1244693 Data available on-line at: https://glad.earthengine.app/view/global-forest-change.
* CHIRPS precipitation data
    * Dataset: CHIRPS Pentad: Climate Hazards Center InfraRed Precipitation With Station Data (Version 2.0 Final) (dataset ID: 'UCSB-CHG/CHIRPS/PENTAD')
    * Variables: Precipitation in mm for a 5 day period (variable name: 'precipitation')
    * Coverage: 44 years (1981-2025)
    * Dataset Provider: UCSB/CHG
    * Citations: 
        * Funk, Chris, Pete Peterson, Martin Landsfeld, Diego Pedreros, James Verdin, Shraddhanand Shukla, Gregory Husak, James Rowland, Laura Harrison, Andrew Hoell & Joel Michaelsen. "The climate hazards infrared precipitation with stations-a new environmental record for monitoring extremes". Scientific Data 2, 150066. doi:10.1038/sdata.2015.66 2015.
* SRTM Elevation data
    * Dataset: NASA SRTM Digital Elevation 30m (dataset ID: 'USGS/SRTMGL1_003')
    * Variables: Elevation in meters (variable name: 'elevation')
    * Coverage: 1 year (2000)
    * Dataset Provider: NASA / USGS / JPL-Caltech
    * Citations:
        * Farr, T.G., Rosen, P.A., Caro, E., Crippen, R., Duren, R., Hensley, S., Kobrick, M., Paller, M., Rodriguez, E., Roth, L., Seal, D., Shaffer, S., Shimada, J., Umland, J., Werner, M., Oskin, M., Burbank, D., and Alsdorf, D.E., 2007, The shuttle radar topography mission: Reviews of Geophysics, v. 45, no. 2, RG2004, at https://doi.org/10.1029/2005RG000183.
* CCNL Nighttime Light data
    * Dataset: CCNL: Consistent and Corrected Nighttime Light Dataset from DMSP-OLS (1992-2013) v1 (dataset ID: 'BNU/FGS/CCNL/v1')
    * Variables: Corrected nighttime light intensity (variable name: 'b1')
    * Coverage: 21 years (1992-2013)
    * Dataset Provider: Beijing Normal University
    * Citations:
        * Zhao,Chenchen, Cao,Xin, Chen,Xuehong, & Cui,Xihong. (2020). A Consistent and Corrected Nighttime Light dataset (CCNL 1992-2013) from DMSP-OLS data (Version 1.0) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.6644980
* GHSL Urbanization data
    * Dataset: GHSL: Degree of Urbanization 1975-2030 V2-0 (P2023A) (dataset ID: 'JRC/GHSL/P2023A/GHS_SMOD_V2-0')
    * Variables: Degree of urbanization (-200: No Data; 10: Water; 11: Very low density rural; 12: Low Density Rural; 13: Rural Cluster; 21: Suburban or peri-urban; 22: Semi-dense urban cluster; 23: Dense urban cluster; 30: Urban centre) (variable name: 'smod_code')
    * Coverage: 55 years (1975-2030)
    * Dataset Provider: EC JRC
    * Citations:
        * Schiavina, Marcello; Melchiorri, Michele; Pesaresi, Martino (2023): GHS-SMOD R2023A - GHS settlement layers, application of the Degree of Urbanisation methodology (stage I) to GHS-POP R2023A and GHS-BUILT-S R2023A, multitemporal (1975-2030). European Commission, Joint Research Centre (JRC) PID: http://data.europa.eu/89h/a0df7a6f-49de-46ea-9bde-563437a6e2ba doi:10.2905/A0DF7A6F-49DE-46EA-9BDE-563437A6E2BA
        * Pesaresi, Martino, Marcello Schiavina, Panagiotis Politis, Sergio Freire, Katarzyna Krasnodebska, Johannes H. Uhl, Alessandra Carioli, et al. (2024). Advances on the Global Human Settlement Layer by Joint Assessment of Earth Observation and Population Survey Data. International Journal of Digital Earth 17(1). doi:10.1080/17538947.2024.2390454.
* Global Critical Infrastructure Spatial Index
    * Dataset: Harmonized Global Critical infrastructure & Index (CISI) (dataset ID: 'projects/sat-io/open-datasets/CISI/global_CISI')
    * Variables: Critical infrastructure spatial index (CISI). [Note: there is also an image collection with the various types of infrastructure derived from Open Street Map at fine granularity, like hospitals, dentist, library, etc. but coverage likely patchy enough not to merit use] (variable name: 'b1')
    * Coverage: extracted 2021 (*not* 1990, as metadata might suggest?)
    * Dataset Provider: authors, Zenodo https://zenodo.org/records/4957647#.Yl5lhFzMJct
    * Citations:
        * Nirandjan, S., Koks, E.E., Ward, P.J. et al. A spatially-explicit harmonized global dataset of critical infrastructure. Sci Data 9, 150 (2022). https://doi.org/10.1038/s41597-022-01218-4 

look for more potential sources: 
* https://gee-community-catalog.org/projects/
* https://unepgrid.ch/en/mapx
* https://pipmaps.worldbank.org/en/data/datatopics/poverty-portal/home
* https://giri.unepgrid.ch
* https://www.nature.com/articles/s41467-022-30727-4
* https://sdgtransformationcenter.org/geospatial
* https://hub.worldpop.org/project/categories?id=14

## Function use - Get results for single point
The following code extracts data for a single point, as specified by the user.

In [ ]:
# Example usage of the generic function for a single point
output = query_dataset_generic(
    site_code=None,
    lat=0, # Change to desired latitude
    lon=0, # Change to desired longitude
    year=2000, # Change to desired year
    dataset_id='dataset_id', # Replaced with the dataset ID
    variable_names=['var1', 'var2', 'var3'], # Replace with the variable names
    buffer_meters=0, # Change to desired buffer distance
    spatial_agg='mean', # Change to preferred aggregation method
    temporal_agg='mean', # Change to preferred temporal aggregation method
    time_window_years=0, # Change to desired time window in years
    closest_subsequent=False, # Change according to whether searching for closest subsequent image (if tempral_agg is 'closest')
)

## Function use - Get results for several points from a CSV file
The following code will extract the data for a set of points in a CSV file. The CSV file should be formatted as having a point (and corresponding year) per line. Each point (row) should have, in the following order, a site code ('site_code'), a country code ('country_code'), latitude coordinate ('lat'), longitude coordinate ('lon'), and year ('year').

In [78]:
# Import data from CSV file (formatted as specified above)
df = pd.read_csv('sites_for_ee.csv') # Replace with your actual CSV file path

In [ ]:
# Initialize an empty list to store results
results = []

# Iterate through every point/row in the dataframe
for index, row in df.iterrows():
    country_code = row['country_code']
    site_code = row['site_code'] 
    lat = row['lat']
    lon = row['lon']
    year = row['year']
    
    # Query the dataset for each point/row
    result = query_dataset_generic(
        site_code=site_code,
        country_code=country_code,
        lat=lat,
        lon=lon,
        year=year,
        dataset_id='dataset_id', # Change to desired dataset ID
        variable_names=['var1', 'var2', 'var3'],  # Change to desired variable names
        buffer_meters=0, # Change to desired buffer distance
        spatial_agg='mean', # Change to preferred aggregation method space around point
        temporal_agg='mean',  # Change to desired temporal aggregation method for time around year
        time_window_years=0, # Change to desired time window in years
        closest_subsequent=False, # Change according to whether searching for closest subsequent image (if tempral_agg is 'closest')
    )

    # Add the original point details to the result
    ordered_result = {
        'site_code': site_code,
        'year': year
    }
    ordered_result.update(result)  
    
    # Append the result to the results list
    results.append(ordered_result)

## Function use - Compile results from all datasets into a data frame
To call all datasets outlined above, compile results into a data frame, and then export this as a CSV file, the code below should be used. Suggested parameters have been set, but can be altered.

In [108]:
# Initialize an empty list to store results
results = []

# Iterate through every point/row in the dataframe
for idx, row in df.iterrows():
    site_code = row['site_code']
    country_code = row['country_code']
    lat = row['lat']
    lon = row['lon']
    year = row['year']

    # Initialize a combined result dictionary
    combined_result = {
        'site_code': site_code,
        'year': year
    }

    ### --- Dataset: CSP Global Human Modification --- ###
    CSP_bands = ee.ImageCollection('CSP/HM/GlobalHumanModification').first().bandNames().getInfo()

    result_CSP = query_dataset_generic(
        site_code=site_code,
        lat=lat,
        lon=lon,
        year=year,
        dataset_id='CSP/HM/GlobalHumanModification',
        variable_names=CSP_bands,
        buffer_meters=250,
        spatial_agg='mean',
        temporal_agg='closest',
        time_window_years=10,
        closest_precedent=False
    )

    prefix = 'CSP'
    combined_result[f'{prefix}_year_diff'] = result_CSP.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_CSP.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_CSP.items()})

    ### --- Dataset: HII --- ###
    HII_bands = ee.ImageCollection('projects/HII/v1/hii').first().bandNames().getInfo()

    result_HII = query_dataset_generic(
        site_code=site_code,
        lat=lat,
        lon=lon,
        year=year,
        dataset_id='projects/HII/v1/hii',
        variable_names=HII_bands,
        buffer_meters=250,
        spatial_agg='mean',
        temporal_agg='closest',
        time_window_years=5,
        closest_precedent=True
    )

    prefix = 'HII'
    combined_result[f'{prefix}_year_diff'] = result_HII.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_HII.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_HII.items()})

    ### --- Dataset: HII Infrastructure --- ###
    HII_infr_bands = ee.ImageCollection('projects/HII/v1/driver/infrastructure').first().bandNames().getInfo()

    result_HII_infr = query_dataset_generic(
        site_code=site_code,
        lat=lat,
        lon=lon,
        year=year,
        dataset_id='projects/HII/v1/driver/infrastructure',
        variable_names=HII_infr_bands,
        buffer_meters=250,
        spatial_agg='mean',
        temporal_agg='closest',
        time_window_years=5,
        closest_precedent=True
    )

    prefix = 'HII_infrastructure'
    combined_result[f'{prefix}_year_diff'] = result_HII_infr.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_HII_infr.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_HII_infr.items()})

    ### --- Dataset: HII Land Use --- ###
    HII_land_bands = ee.ImageCollection('projects/HII/v1/driver/land_use').first().bandNames().getInfo()

    result_HII_land = query_dataset_generic(
        site_code=site_code,
        lat=lat,
        lon=lon,
        year=year,
        dataset_id='projects/HII/v1/driver/land_use',
        variable_names=HII_land_bands,
        buffer_meters=250,
        spatial_agg='mean',
        temporal_agg='closest',
        time_window_years=5,
        closest_precedent=True
    )

    prefix = 'HII_land_use'
    combined_result[f'{prefix}_year_diff'] = result_HII_land.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_HII_land.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_HII_land.items()})

    ### --- Dataset: HII Population Density --- ###
    HII_pop_bands = ee.ImageCollection('projects/HII/v1/driver/population_density').first().bandNames().getInfo()

    result_HII_pop = query_dataset_generic(
        site_code=site_code,
        lat=lat,
        lon=lon,
        year=year,
        dataset_id='projects/HII/v1/driver/population_density',
        variable_names=HII_pop_bands,
        buffer_meters=250,
        spatial_agg='mean',
        temporal_agg='closest',
        time_window_years=5,
        closest_precedent=True
    )

    prefix = 'HII_pop'
    combined_result[f'{prefix}_year_diff'] = result_HII_pop.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_HII_pop.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_HII_pop.items()})

    ### --- Dataset: HII Power --- ###
    HII_power_bands = ee.ImageCollection('projects/HII/v1/driver/power').first().bandNames().getInfo()

    result_HII_power = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='projects/HII/v1/driver/power',
        variable_names=HII_power_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=5, 
        closest_precedent=True
    )

    prefix = 'HII_power'
    combined_result[f'{prefix}_year_diff'] = result_HII_power.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_HII_power.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_HII_power.items()})

    ### --- Dataset: HII Railways --- ###
    HII_railway_bands = ee.ImageCollection('projects/HII/v1/driver/railways').first().bandNames().getInfo()

    result_HII_railway = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='projects/HII/v1/driver/railways',
        variable_names=HII_railway_bands,
        buffer_meters=250, 
        spatial_agg='mean',
        temporal_agg='closest',
        time_window_years=5, 
        closest_precedent=True
    )

    prefix = 'HII_railways'
    combined_result[f'{prefix}_year_diff'] = result_HII_railway.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_HII_railway.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_HII_railway.items()})

    ### --- Dataset: HII Roads --- ###
    HII_road_bands = ee.ImageCollection('projects/HII/v1/driver/roads').first().bandNames().getInfo()

    result_HII_road = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='projects/HII/v1/driver/roads',
        variable_names=HII_road_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=5, 
        closest_precedent=True
    )

    prefix = 'HII_roads'
    combined_result[f'{prefix}_year_diff'] = result_HII_road.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_HII_road.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_HII_road.items()})

    ### --- Dataset: HII Water --- ###
    HII_water_bands = ee.ImageCollection('projects/HII/v1/driver/water').first().bandNames().getInfo()

    result_HII_water = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='projects/HII/v1/driver/water',
        variable_names=HII_water_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=5, 
        closest_precedent=True
    )

    prefix = 'HII_water'
    combined_result[f'{prefix}_year_diff'] = result_HII_water.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_HII_water.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_HII_water.items()})

    ### --- Dataset: GHSL --- ###
    GHSL_C_bands = ee.ImageCollection('JRC/GHSL/P2023A/GHS_BUILT_C').first().bandNames().getInfo()

    # Note: putting spatial_agg as 'median' since codes are categorical responses!
    result_GHSL_C = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='JRC/GHSL/P2023A/GHS_BUILT_C',
        variable_names=GHSL_C_bands,
        buffer_meters=250, 
        spatial_agg='median', 
        temporal_agg='closest',
        time_window_years=10,
        closest_precedent=False
    )

    prefix = 'GHSL_characteristics'
    combined_result[f'{prefix}_year_diff'] = result_GHSL_C.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_GHSL_C.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_GHSL_C.items()})

    GHSL_S_bands = ee.ImageCollection('JRC/GHSL/P2023A/GHS_BUILT_S').first().bandNames().getInfo()

    result_GHSL_S = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='JRC/GHSL/P2023A/GHS_BUILT_S',
        variable_names=GHSL_S_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=10,
        closest_precedent=False
    )

    prefix = 'GHSL_surface'
    combined_result[f'{prefix}_year_diff'] = result_GHSL_S.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_GHSL_S.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_GHSL_S.items()})

    ### --- Dataset: World Pop --- ###
    wp_bands = ee.ImageCollection('WorldPop/GP/100m/pop').first().bandNames().getInfo()

    result_wp = query_dataset_generic(
        site_code=site_code, 
        country_code=country_code,
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='WorldPop/GP/100m/pop',
        variable_names=wp_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=5, 
        closest_precedent=False
    )

    prefix = 'WorldPop'
    combined_result[f'{prefix}_year_diff'] = result_wp.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_wp.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_wp.items()})

    # 500 m
    result_wp_500m = query_dataset_generic(
        site_code=site_code, 
        country_code=country_code,
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='WorldPop/GP/100m/pop',
        variable_names=wp_bands,
        buffer_meters=500, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=5, 
        closest_precedent=False
    )

    combined_result[f'{prefix}_year_diff'] = result_wp_500m.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_wp_500m.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_wp_500m.items()})

    # 1 km
    result_wp_1k = query_dataset_generic(
        site_code=site_code, 
        country_code=country_code,
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='WorldPop/GP/100m/pop',
        variable_names=wp_bands,
        buffer_meters=1000, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=5, 
        closest_precedent=False
    )

    combined_result[f'{prefix}_year_diff'] = result_wp_1k.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_wp_1k.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_wp_1k.items()})

    # 5 km
    result_wp_5k = query_dataset_generic(
        site_code=site_code, 
        country_code=country_code,
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='WorldPop/GP/100m/pop',
        variable_names=wp_bands,
        buffer_meters=5000, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=5, 
        closest_precedent=False
    )

    combined_result[f'{prefix}_year_diff'] = result_wp_5k.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_wp_5k.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_wp_5k.items()})

    ### --- Dataset: GPW Population Density --- ###
    gpw_bands = ee.ImageCollection('CIESIN/GPWv411/GPW_Population_Density').first().bandNames().getInfo()

    result_gpw = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='CIESIN/GPWv411/GPW_Population_Density',
        variable_names=gpw_bands,
        buffer_meters=250,
        spatial_agg='mean',
        temporal_agg='closest',
        time_window_years=5,
        closest_precedent=True
    )

    prefix = 'GPW'
    combined_result[f'{prefix}_year_diff'] = result_gpw.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_gpw.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_gpw.items()})

    ### --- Dataset: Oxford MAP --- ###
    map_bands = ee.Image('Oxford/MAP/accessibility_to_cities_2015_v1_0').bandNames().getInfo()

    result_map = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='Oxford/MAP/accessibility_to_cities_2015_v1_0',
        variable_names=map_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=10, 
        closest_precedent=False
    )

    prefix = 'MAP'
    combined_result[f'{prefix}_year_diff'] = result_map.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_map.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_map.items()})

    result_map_1k = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='Oxford/MAP/accessibility_to_cities_2015_v1_0',
        variable_names=map_bands,
        buffer_meters=1000, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=10, 
        closest_precedent=False
    )

    combined_result[f'{prefix}_year_diff'] = result_map_1k.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_map_1k.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_map_1k.items()})

    result_map_5k = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='Oxford/MAP/accessibility_to_cities_2015_v1_0',
        variable_names=map_bands,
        buffer_meters=5000, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=10, 
        closest_precedent=False
    )

    combined_result[f'{prefix}_year_diff'] = result_map_5k.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_map_5k.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_map_5k.items()})

    ### --- Dataset: RAI Multiplier --- ###
    rai_mult_bands = ee.Image('projects/sat-io/open-datasets/RAI/raimultiplier').bandNames().getInfo()

    result_rai_mult = query_dataset_generic(
        site_code=site_code, 
        lat=lat, lon=lon, 
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/raimultiplier',
        variable_names=rai_mult_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )
    
    prefix = 'RAI_multiplier'
    combined_result[f'{prefix}_year_diff'] = result_rai_mult.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_mult.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_rai_mult.items()})

    result_rai_mult_1k = query_dataset_generic(
        site_code=site_code, 
        lat=lat, lon=lon, 
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/raimultiplier',
        variable_names=rai_mult_bands,
        buffer_meters=1000, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )
    
    combined_result[f'{prefix}_year_diff'] = result_rai_mult_1k.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_mult_1k.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_rai_mult_1k.items()})

    result_rai_mult_0 = query_dataset_generic(
        site_code=site_code, 
        lat=lat, lon=lon, 
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/raimultiplier',
        variable_names=rai_mult_bands,
        buffer_meters=0, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )
    
    combined_result[f'{prefix}_year_diff'] = result_rai_mult_0.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_mult_0.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_rai_mult_0.items()})

    ### --- Dataset: RAI Rural Pop --- ###
    rai_ruralpop_bands = ee.Image('projects/sat-io/open-datasets/RAI/ruralpop').bandNames().getInfo()

    result_rai_ruralpop = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon,
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/ruralpop',
        variable_names=rai_ruralpop_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )

    prefix = 'RAI_ruralpop'
    combined_result[f'{prefix}_year_diff'] = result_rai_ruralpop.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_ruralpop.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_rai_ruralpop.items()})

    result_rai_ruralpop_1k = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon,
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/ruralpop',
        variable_names=rai_ruralpop_bands,
        buffer_meters=1000, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )

    combined_result[f'{prefix}_year_diff'] = result_rai_ruralpop_1k.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_ruralpop_1k.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_rai_ruralpop_1k.items()})

    result_rai_ruralpop_5k = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon,
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/ruralpop',
        variable_names=rai_ruralpop_bands,
        buffer_meters=5000, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )

    combined_result[f'{prefix}_year_diff'] = result_rai_ruralpop_5k.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_ruralpop_5k.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_rai_ruralpop_5k.items()})

    result_rai_ruralpop_0 = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon,
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/ruralpop',
        variable_names=rai_ruralpop_bands,
        buffer_meters=0, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )

    combined_result[f'{prefix}_year_diff'] = result_rai_ruralpop_0.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_ruralpop_0.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_rai_ruralpop_0.items()})

    ### --- Dataset: RAI Rural Pop with Access --- ###
    rai_ruralacc_bands = ee.Image('projects/sat-io/open-datasets/RAI/ruralpopaccess').bandNames().getInfo()

    result_rai_ruralacc = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/ruralpopaccess',
        variable_names=rai_ruralacc_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0,
        closest_precedent=True
    )

    prefix = 'RAI_ruralpopaccess'
    combined_result[f'{prefix}_year_diff'] = result_rai_ruralacc.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_ruralacc.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_rai_ruralacc.items()})

    result_rai_ruralacc_1k = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/ruralpopaccess',
        variable_names=rai_ruralacc_bands,
        buffer_meters=1000, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0,
        closest_precedent=True
    )

    combined_result[f'{prefix}_year_diff'] = result_rai_ruralacc_1k.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_ruralacc_1k.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_rai_ruralacc_1k.items()})

    result_rai_ruralacc_5k = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/ruralpopaccess',
        variable_names=rai_ruralacc_bands,
        buffer_meters=5000, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0,
        closest_precedent=True
    )

    combined_result[f'{prefix}_year_diff'] = result_rai_ruralacc_5k.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_ruralacc_5k.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_rai_ruralacc_5k.items()})

    result_rai_ruralacc_0 = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/ruralpopaccess',
        variable_names=rai_ruralacc_bands,
        buffer_meters=0, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0,
        closest_precedent=True
    )

    combined_result[f'{prefix}_year_diff'] = result_rai_ruralacc_0.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_ruralacc_0.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_rai_ruralacc_0.items()})

    ### --- Dataset: GDP per Capita --- ###
    
    result_gdp = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='projects/earthengine-legacy/assets/projects/sat-io/open-datasets/GRIDDED_HDI_GDP/GDP_per_capita_PPP_1990_2015_v2',
        variable_names=[], 
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )

    prefix = 'GDP_PPP'
    combined_result[f'{prefix}_year_diff'] = result_gdp.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_gdp.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_gdp.items()})

    ### --- Dataset: HDI --- ###

    result_hdi = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='projects/sat-io/open-datasets/GRIDDED_HDI_GDP/HDI_1990_2015_v2',
        variable_names=[],
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )

    prefix = 'HDI'
    combined_result[f'{prefix}_year_diff'] = result_hdi.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_hdi.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_hdi.items()})

    ### --- Dataset: TERRACLIMATE --- ###
    terraclimate_bands = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').first().bandNames().getInfo()

    result_terra = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='IDAHO_EPSCOR/TERRACLIMATE',
        variable_names=terraclimate_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='mean',
        time_window_years=0, 
        closest_precedent=True
    )
    
    prefix = 'TERRACLIMATE'
    combined_result[f'{prefix}_year_diff'] = result_terra.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_terra.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_terra.items()})

    ### --- Dataset: GRACE MASCON --- ###
    grace_bands = ee.ImageCollection('NASA/GRACE/MASS_GRIDS_V04/MASCON_CRI').first().bandNames().getInfo()

    result_grace = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='NASA/GRACE/MASS_GRIDS_V04/MASCON_CRI',
        variable_names=grace_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='mean',
        time_window_years=0, 
        closest_precedent=True
    )

    prefix = 'GRACE'
    combined_result[f'{prefix}_year_diff'] = result_grace.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_grace.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_grace.items()})

    ### --- Dataset: SPEI --- ###
    spei_bands = ee.ImageCollection('CSIC/SPEI/2_10').first().bandNames().getInfo()

    result_spei = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='CSIC/SPEI/2_10',
        variable_names=spei_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )

    prefix = 'SPEI'
    combined_result[f'{prefix}_year_diff'] = result_spei.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_spei.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_spei.items()})

    ### --- Dataset: Hansen Forest Change --- ###
    hansen_bands = ee.Image('UMD/hansen/global_forest_change_2024_v1_12').bandNames().getInfo()

    result_hansen = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='UMD/hansen/global_forest_change_2024_v1_12',
        variable_names=hansen_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )

    prefix = 'Hansen'
    combined_result[f'{prefix}_year_diff'] = result_hansen.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_hansen.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_hansen.items()})

    ### --- Dataset: CHIRPS Precipitation --- ###
    chirps_bands = ee.ImageCollection('UCSB-CHG/CHIRPS/PENTAD').first().bandNames().getInfo()

    result_chirps = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='UCSB-CHG/CHIRPS/PENTAD',
        variable_names=chirps_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='mean',
        time_window_years=0, 
        closest_precedent=True
    )

    prefix = 'CHIRPS'
    combined_result[f'{prefix}_year_diff'] = result_chirps.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_chirps.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_chirps.items()})

    ### --- Dataset: SRTM Elevation --- ###
    srtm_bands = ee.Image('USGS/SRTMGL1_003').bandNames().getInfo()

    result_srtm = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='USGS/SRTMGL1_003',
        variable_names=srtm_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=False
    )
    
    prefix = 'SRTM'
    combined_result[f'{prefix}_year_diff'] = result_srtm.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_srtm.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_srtm.items()})

    ### --- Dataset: CCNL Nightime Light --- ###
    ccnl_bands = ee.ImageCollection('BNU/FGS/CCNL/v1').first().bandNames().getInfo()

    result_ccnl = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='BNU/FGS/CCNL/v1',
        variable_names=ccnl_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=15, 
        closest_precedent=True
    )
    
    prefix = 'CCNL'
    combined_result[f'{prefix}_year_diff'] = result_ccnl.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_ccnl.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_ccnl.items()})

    ### --- Dataset: GHSL Urbanization --- ###
    ghsl_urban_bands = ee.ImageCollection('JRC/GHSL/P2023A/GHS_SMOD_V2-0').first().bandNames().getInfo()

    # Note: spatial aggregation set to 'median' since codes are categorical
    result_ghsl_urban = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='JRC/GHSL/P2023A/GHS_SMOD_V2-0',
        variable_names=ghsl_urban_bands,
        buffer_meters=250, 
        spatial_agg='median', 
        temporal_agg='closest',
        time_window_years=5, 
        closest_precedent=True
    )
    
    prefix = 'GHSL'
    combined_result[f'{prefix}_year_diff'] = result_ghsl_urban.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_ghsl_urban.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_ghsl_urban.items()})

    ### --- Dataset: Global Critical Infrastructure --- ###
    cisi_bands = ee.Image('projects/sat-io/open-datasets/CISI/global_CISI').bandNames().getInfo()

    result_cisi = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='projects/sat-io/open-datasets/CISI/global_CISI',
        variable_names=cisi_bands,
        buffer_meters=250, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=5, 
        closest_precedent=True
    )
    
    prefix = 'CISI'
    combined_result[f'{prefix}_year_diff'] = result_cisi.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_cisi.pop(k, None)
    combined_result.update({f"{prefix}_{k}": v for k, v in result_cisi.items()})

    ### --- Append final result --- ###
    results.append(combined_result)

# Output to DataFrame and CSV
df_results = pd.DataFrame(results)
df_results.to_csv("gee_output.csv", index=False)

Querying image collection 'CSP/HM/GlobalHumanModification' at (-1.940887, 147.102236), year 2018 with variables ['gHM']...
Querying image collection 'projects/HII/v1/hii' at (-1.940887, 147.102236), year 2018 with variables ['hii']...
Querying image collection 'projects/HII/v1/driver/infrastructure' at (-1.940887, 147.102236), year 2018 with variables ['hii_infrastucture_driver']...
Querying image collection 'projects/HII/v1/driver/land_use' at (-1.940887, 147.102236), year 2018 with variables ['hii_landuse_driver']...
Querying image collection 'projects/HII/v1/driver/population_density' at (-1.940887, 147.102236), year 2018 with variables ['hii_popdens_driver']...
Querying image collection 'projects/HII/v1/driver/power' at (-1.940887, 147.102236), year 2018 with variables ['hii_power_driver']...
Querying image collection 'projects/HII/v1/driver/railways' at (-1.940887, 147.102236), year 2018 with variables ['hii_railway_driver']...
Querying image collection 'projects/HII/v1/driver/roa

In [106]:
query_dataset_generic(
        site_code='te', 
        lat=9.626, 
        lon=78.448,
        year=2017,
        dataset_id='projects/sat-io/open-datasets/RAI/ruralpopaccess',
        variable_names=rai_ruralacc_bands,
        buffer_meters=0, 
        spatial_agg='min', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )

Querying static image dataset 'projects/sat-io/open-datasets/RAI/ruralpopaccess' at (9.626, 78.448), year 2017 with variables ['population']...


{'population_buffer0_spatialmin_value': 261.2066659927368,
 'population_buffer0_spatialmin_date': 'unknown',
 'year_diff': None}

In [ ]:
# Initialize an empty list to store results
results1 = []

# Iterate through every point/row in the dataframe
for idx, row in df.iterrows():
    site_code = row['site_code']
    country_code = row['country_code']
    lat = row['lat']
    lon = row['lon']
    year = row['year']

    # Initialize a combined result dictionary
    combined_result1 = {
        'site_code': site_code,
        'lat': lat,
        'lon': lon,
        'year': year
    }

    ### --- Dataset: World Pop --- ###
    wp_bands = ee.ImageCollection('WorldPop/GP/100m/pop').first().bandNames().getInfo()

    result_wp = query_dataset_generic(
        site_code=site_code, 
        country_code=country_code,
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='WorldPop/GP/100m/pop',
        variable_names=wp_bands,
        buffer_meters=500, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=5, 
        closest_precedent=False
    )

    prefix = 'WorldPop'
    combined_result1[f'{prefix}_year_diff'] = result_wp.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_wp.pop(k, None)
    combined_result1.update({f"{prefix}_{k}": v for k, v in result_wp.items()})

    ### --- Dataset: RAI Rural Pop --- ###
    rai_ruralpop_bands = ee.Image('projects/sat-io/open-datasets/RAI/ruralpop').bandNames().getInfo()

    result_rai_ruralpop_sum = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon,
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/ruralpop',
        variable_names=rai_ruralpop_bands,
        buffer_meters=5000, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )

    prefix = 'RAI_ruralpop'
    combined_result1[f'{prefix}_year_diff'] = result_rai_ruralpop_sum.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_ruralpop_sum.pop(k, None)
    combined_result1.update({f"{prefix}_{k}": v for k, v in result_rai_ruralpop_sum.items()})

    result_rai_ruralpop_5km = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon,
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/ruralpop',
        variable_names=rai_ruralpop_bands,
        buffer_meters=5000, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )

    combined_result1[f'{prefix}_year_diff'] = result_rai_ruralpop_5km.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_ruralpop_5km.pop(k, None)
    combined_result1.update({f"{prefix}_{k}": v for k, v in result_rai_ruralpop_5km.items()})

    ### --- Dataset: RAI Rural Pop with Access --- ###
    rai_ruralacc_bands = ee.Image('projects/sat-io/open-datasets/RAI/ruralpopaccess').bandNames().getInfo()

    result_rai_ruralacc_sum = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/ruralpopaccess',
        variable_names=rai_ruralacc_bands,
        buffer_meters=5000, 
        spatial_agg='sum', 
        temporal_agg='closest',
        time_window_years=0,
        closest_precedent=True
    )

    prefix = 'RAI_ruralpopaccess'
    combined_result1[f'{prefix}_year_diff'] = result_rai_ruralacc_sum.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_ruralacc_sum.pop(k, None)
    combined_result1.update({f"{prefix}_{k}": v for k, v in result_rai_ruralacc_sum.items()})

    result_rai_ruralacc_5km = query_dataset_generic(
        site_code=site_code, 
        lat=lat, 
        lon=lon, 
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/ruralpopaccess',
        variable_names=rai_ruralacc_bands,
        buffer_meters=5000, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0,
        closest_precedent=True
    )

    combined_result1[f'{prefix}_year_diff'] = result_rai_ruralacc_5km.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_ruralacc_5km.pop(k, None)
    combined_result1.update({f"{prefix}_{k}": v for k, v in result_rai_ruralacc_5km.items()})

    ### --- Dataset: RAI Multiplier --- ###
    rai_mult_bands = ee.Image('projects/sat-io/open-datasets/RAI/raimultiplier').bandNames().getInfo()

    result_rai_mult_5km = query_dataset_generic(
        site_code=site_code, 
        lat=lat, lon=lon, 
        year=year,
        dataset_id='projects/sat-io/open-datasets/RAI/raimultiplier',
        variable_names=rai_mult_bands,
        buffer_meters=5000, 
        spatial_agg='mean', 
        temporal_agg='closest',
        time_window_years=0, 
        closest_precedent=True
    )
    
    combined_result1[f'{prefix}_year_diff'] = result_rai_mult_5km.pop('year_diff', None)
    for k in ['site_code', 'target_lat', 'target_lon', 'target_year']:
        result_rai_mult_5km.pop(k, None)
    combined_result1.update({f"{prefix}_{k}": v for k, v in result_rai_mult_5km.items()})

    ### --- Append final result --- ###
    results1.append(combined_result1)

# Output to DataFrame and CSV
df_results1 = pd.DataFrame(results1)
df_results1.to_csv("your_output_file2.csv", index=False) # Change to desired output file name

Querying image collection 'WorldPop/GP/100m/pop' at (-1.940887, 147.102236), year 2018 with variables ['population']...
Querying static image dataset 'projects/sat-io/open-datasets/RAI/ruralpop' at (-1.940887, 147.102236), year 2018 with variables ['population']...
Querying static image dataset 'projects/sat-io/open-datasets/RAI/ruralpop' at (-1.940887, 147.102236), year 2018 with variables ['population']...
Querying static image dataset 'projects/sat-io/open-datasets/RAI/ruralpopaccess' at (-1.940887, 147.102236), year 2018 with variables ['population']...
Querying static image dataset 'projects/sat-io/open-datasets/RAI/ruralpopaccess' at (-1.940887, 147.102236), year 2018 with variables ['population']...
Querying static image dataset 'projects/sat-io/open-datasets/RAI/raimultiplier' at (-1.940887, 147.102236), year 2018 with variables ['b1']...
Querying image collection 'WorldPop/GP/100m/pop' at (-17.24675, 13.75184), year 2022 with variables ['population']...
Querying static image d